# 📋 Extração de Dados de Folha de Ponto - Plano TODO Atualizado

## 🎯 STATUS ATUAL: PONTO 1.1 CONCLUÍDO ✅

### ✅ **FASE 1: ANÁLISE E ESTRUTURAÇÃO DOS DADOS**

#### ✅ **1.1 Identificação das Informações da Folha** - **CONCLUÍDO**
- ✅ **Dados do Cabeçalho da Empresa**
  - ✅ Extrair nome da empresa
  - ✅ Extrair CNPJ da empresa
  - ✅ Extrair logo/imagem da empresa
  - ❌ ~~Extrair endereço da empresa~~ (não existe no PDF)

- ✅ **Dados do Funcionário** 
  - ✅ Extrair nome completo do funcionário
  - ✅ Extrair CPF do funcionário
  - ❌ ~~Extrair PIS/PASEP~~ (não existe no PDF)
  - ❌ ~~Extrair matrícula/código do funcionário~~ (não existe no PDF)
  - ❌ ~~Extrair cargo/função~~ (não existe no PDF)
  - ❌ ~~Extrair departamento/setor~~ (não existe no PDF)

- ✅ **Dados do Período**
  - ✅ Extrair mês de referência
  - ✅ Extrair ano de referência
  - ✅ Identificar data de início do período
  - ✅ Identificar data de fim do período

#### 🔄 **1.2 Mapeamento da Tabela de Registros de Ponto** - **PRÓXIMO**
- [ ] **Estrutura da Tabela Principal**
  - [ ] Identificar colunas de data
  - [ ] Identificar colunas de entrada manhã
  - [ ] Identificar colunas de saída manhã
  - [ ] Identificar colunas de entrada tarde
  - [ ] Identificar colunas de saída tarde
  - [ ] Identificar colunas de horas trabalhadas
  - [ ] Identificar colunas de observações/justificativas

### 📊 **RESULTADOS OBTIDOS:**
- **Taxa de sucesso: 100%** (10/10 campos existentes extraídos)
- **Arquivos gerados:** `folha_ponto_dados.json`
- **Classe funcional:** `TimesheetHeaderExtractor`

### 🚀 **PRÓXIMOS PASSOS:**
1. **Implementar ponto 1.2:** Extração da tabela de registros de ponto
2. **Desenvolver cálculos:** Horas trabalhadas, extras, faltas
3. **Criar validações:** Consistência de horários
4. **Implementar relatórios:** CSV, Excel, PDF

In [1]:
# Imports necessários
import fitz  # PyMuPDF
import re
import pandas as pd
from datetime import datetime
from typing import Dict, List, Optional, Tuple
import json
from pathlib import Path

# Configurações
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("✅ Bibliotecas importadas com sucesso!")
print(f"📚 PyMuPDF versão: {fitz.__version__ if hasattr(fitz, '__version__') else 'N/A'}")
print(f"🐼 Pandas versão: {pd.__version__}")

# Caminho para o arquivo de teste
PDF_PATH = "../../test_sample/Ficha_Ponto_Simplificada_André_Luis.pdf"

✅ Bibliotecas importadas com sucesso!
📚 PyMuPDF versão: 1.26.3
🐼 Pandas versão: 2.3.1


In [9]:
class TimesheetHeaderExtractor:
    """
    Extrator limpo e funcional para dados de cabeçalho de folhas de ponto.
    Extrai apenas os campos que realmente existem no PDF.
    """
    
    def extract_header_data(self, pdf_path: str) -> Dict:
        """
        Extrai dados de cabeçalho da folha de ponto.
        
        Returns:
            Dict: Dados extraídos organizados por categoria
        """
        try:
            doc = fitz.open(pdf_path)
            page = doc[0]
            text = page.get_text()
            lines = [line.strip() for line in text.split('\n') if line.strip()]
            
            result = {
                'company': self._extract_company_data(lines, page),
                'employee': self._extract_employee_data(lines, text),
                'period': self._extract_period_data(lines, text)
            }
            
            doc.close()
            return result
            
        except Exception as e:
            print(f"❌ Erro ao extrair dados: {e}")
            return {}
    
    def _extract_company_data(self, lines: List[str], page) -> Dict:
        """Extrai dados da empresa (apenas campos existentes)."""
        company_data = {
            'name': lines[0] if lines else None,
            'cnpj': None,
            'logo_detected': len(page.get_images()) > 0
        }
        
        # Buscar CNPJ nas linhas
        for line in lines:
            if re.search(r'\d{2}\.\d{3}\.\d{3}/\d{4}-\d{2}', line):
                company_data['cnpj'] = line
                break
                
        return company_data
    
    def _extract_employee_data(self, lines: List[str], full_text: str) -> Dict:
        """Extrai dados do funcionário (apenas campos existentes)."""
        employee_data = {
            'name': None,
            'cpf': None
        }
        
        # Nome: procurar nome em maiúsculas (ex: "ANDRE LUIS DE MORAES DA ROSA")
        for line in lines:
            if (re.match(r'^[A-Z\s]{10,}$', line) and 
                line not in ['SCALA TRANSPORTE E ADMINISTRACAO LTDA.'] and
                not any(word in line for word in ['LTDA', 'S/A', 'ME', 'FICHA', 'PONTO'])):
                employee_data['name'] = line
                break
        
        # CPF: procurar padrão XXX.XXX.XXX-XX
        for line in lines:
            if re.search(r'\d{3}\.\d{3}\.\d{3}-\d{2}', line):
                employee_data['cpf'] = line
                break
        
        return employee_data
    
    def _extract_period_data(self, lines: List[str], full_text: str) -> Dict:
        """Extrai dados do período de referência."""
        period_data = {
            'start_date': None,
            'end_date': None,
            'month': None,
            'year': None,
            'reference_period': None
        }
        
        # Procurar linha com período: "Período: DD/MM/YYYY à DD/MM/YYYY"
        for line in lines:
            if 'Período:' in line:
                dates = re.findall(r'\d{2}/\d{2}/\d{4}', line)
                if len(dates) >= 2:
                    period_data['start_date'] = dates[0]
                    period_data['end_date'] = dates[1]
                    
                    # Extrair mês e ano da data de início
                    start_parts = dates[0].split('/')
                    period_data['month'] = int(start_parts[1])
                    period_data['year'] = int(start_parts[2])
                    
                    # Criar período de referência formatado
                    months = ['', 'Janeiro', 'Fevereiro', 'Março', 'Abril', 'Maio', 'Junho',
                             'Julho', 'Agosto', 'Setembro', 'Outubro', 'Novembro', 'Dezembro']
                    period_data['reference_period'] = f"{months[period_data['month']]}/{period_data['year']}"
                break
        
        return period_data

# Executar extração
print("🎯 EXTRAÇÃO DE DADOS DA FOLHA DE PONTO")
print("=" * 50)

extractor = TimesheetHeaderExtractor()
extracted_data = extractor.extract_header_data(PDF_PATH)

# Exibir resultados organizados
for category, data in extracted_data.items():
    print(f"\n📋 {category.upper()}:")
    print("-" * 25)
    for key, value in data.items():
        status = "✅" if value else "❌"
        field_name = key.replace('_', ' ').title()
        print(f"{status} {field_name}: {value}")

# Estatísticas finais
print(f"\n📊 RESUMO DA EXTRAÇÃO:")
print("=" * 30)
all_fields = []
extracted_fields = []

for category, data in extracted_data.items():
    for key, value in data.items():
        all_fields.append(f"{category}.{key}")
        if value:
            extracted_fields.append(f"{category}.{key}")

success_rate = (len(extracted_fields) / len(all_fields)) * 100
print(f"📈 Campos extraídos: {len(extracted_fields)}/{len(all_fields)}")
print(f"🎯 Taxa de sucesso: {success_rate:.1f}%")

print(f"\n✅ CAMPOS EXTRAÍDOS COM SUCESSO:")
for field in extracted_fields:
    print(f"   • {field.replace('_', ' ').title()}")

print(f"\n🎉 PONTO 1.1 DO PLANO TODO CONCLUÍDO!")
print("✨ Extração de dados de cabeçalho funcionando perfeitamente.")

🎯 EXTRAÇÃO DE DADOS DA FOLHA DE PONTO

📋 COMPANY:
-------------------------
✅ Name: SCALA TRANSPORTE E ADMINISTRACAO LTDA.
✅ Cnpj: 88.501.093/0001-89
✅ Logo Detected: True

📋 EMPLOYEE:
-------------------------
✅ Name: ANDRE LUIS DE MORAES DA ROSA
✅ Cpf: 019.450.890-08

📋 PERIOD:
-------------------------
✅ Start Date: 21/05/2025
✅ End Date: 20/06/2025
✅ Month: 5
✅ Year: 2025
✅ Reference Period: Maio/2025

📊 RESUMO DA EXTRAÇÃO:
📈 Campos extraídos: 10/10
🎯 Taxa de sucesso: 100.0%

✅ CAMPOS EXTRAÍDOS COM SUCESSO:
   • Company.Name
   • Company.Cnpj
   • Company.Logo Detected
   • Employee.Name
   • Employee.Cpf
   • Period.Start Date
   • Period.End Date
   • Period.Month
   • Period.Year
   • Period.Reference Period

🎉 PONTO 1.1 DO PLANO TODO CONCLUÍDO!
✨ Extração de dados de cabeçalho funcionando perfeitamente.


In [10]:
# Exportar dados para uso em outros sistemas
print("💾 EXPORTANDO DADOS EXTRAÍDOS")
print("=" * 35)

# 1. Criar DataFrame estruturado
export_data = []
for category, data in extracted_data.items():
    for key, value in data.items():
        export_data.append({
            'categoria': category,
            'campo': key.replace('_', ' ').title(),
            'valor': str(value) if value is not None else '',
            'tipo': type(value).__name__
        })

df_export = pd.DataFrame(export_data)
print("📊 DataFrame criado:")
print(df_export.to_string(index=False))

# 2. Salvar em JSON para integração com outros sistemas
json_output = {
    'arquivo_processado': PDF_PATH,
    'data_extracao': datetime.now().isoformat(),
    'dados_extraidos': extracted_data,
    'estatisticas': {
        'total_campos': len(all_fields),
        'campos_extraidos': len(extracted_fields), 
        'taxa_sucesso': f"{success_rate:.1f}%"
    }
}

with open('folha_ponto_dados.json', 'w', encoding='utf-8') as f:
    json.dump(json_output, f, ensure_ascii=False, indent=2, default=str)

print(f"\n✅ Dados salvos em: folha_ponto_dados.json")
print(f"✅ DataFrame disponível na variável: df_export")
print(f"✅ Dados estruturados disponíveis na variável: extracted_data")

print(f"\n🔧 PRÓXIMOS PASSOS:")
print("1️⃣ Implementar ponto 1.2: Mapeamento da Tabela de Registros")
print("2️⃣ Desenvolver extração dos horários de ponto")
print("3️⃣ Criar validações de dados")
print("4️⃣ Implementar cálculos de horas trabalhadas")

💾 EXPORTANDO DADOS EXTRAÍDOS
📊 DataFrame criado:
categoria            campo                                  valor tipo
  company             Name SCALA TRANSPORTE E ADMINISTRACAO LTDA.  str
  company             Cnpj                     88.501.093/0001-89  str
  company    Logo Detected                                   True bool
 employee             Name           ANDRE LUIS DE MORAES DA ROSA  str
 employee              Cpf                         019.450.890-08  str
   period       Start Date                             21/05/2025  str
   period         End Date                             20/06/2025  str
   period            Month                                      5  int
   period             Year                                   2025  int
   period Reference Period                              Maio/2025  str

✅ Dados salvos em: folha_ponto_dados.json
✅ DataFrame disponível na variável: df_export
✅ Dados estruturados disponíveis na variável: extracted_data

🔧 PRÓXIMOS PASSOS:


In [11]:
# Limpeza de variáveis desnecessárias e status final
print("🧹 LIMPEZA E STATUS FINAL")
print("=" * 40)

# Limpar variáveis de versões antigas que não são mais necessárias
variables_to_clean = [
    'consolidated_data', 'df_results', 'header_data', 'final_data', 
    'final_extractor', 'improved_data', 'improved_extractor',
    'id_patterns', 'name_patterns', 'matches', 'icon', 'status'
]

cleaned_count = 0
for var in variables_to_clean:
    if var in globals():
        del globals()[var]
        cleaned_count += 1

print(f"✅ {cleaned_count} variáveis antigas removidas")

# Variáveis importantes mantidas
important_vars = {
    'extractor': 'Classe principal de extração',
    'extracted_data': 'Dados extraídos estruturados',
    'df_export': 'DataFrame para análise',
    'PDF_PATH': 'Caminho do arquivo PDF'
}

print(f"\n📋 VARIÁVEIS IMPORTANTES MANTIDAS:")
for var, desc in important_vars.items():
    if var in globals():
        print(f"   ✅ {var}: {desc}")

print(f"\n🎉 PONTO 1.1 FINALIZADO COM SUCESSO!")
print("=" * 40)
print("📊 10/10 campos extraídos (100% sucesso)")
print("🔧 Código limpo e otimizado") 
print("💾 Dados exportados em JSON")
print("🚀 Pronto para implementar ponto 1.2")

print(f"\n📝 RESUMO EXECUTIVO:")
print(f"• Empresa: {extracted_data['company']['name']}")
print(f"• Funcionário: {extracted_data['employee']['name']}")
print(f"• Período: {extracted_data['period']['reference_period']}")
print(f"• Status: Dados de cabeçalho extraídos com sucesso")

🧹 LIMPEZA E STATUS FINAL
✅ 12 variáveis antigas removidas

📋 VARIÁVEIS IMPORTANTES MANTIDAS:
   ✅ extractor: Classe principal de extração
   ✅ extracted_data: Dados extraídos estruturados
   ✅ df_export: DataFrame para análise
   ✅ PDF_PATH: Caminho do arquivo PDF

🎉 PONTO 1.1 FINALIZADO COM SUCESSO!
📊 10/10 campos extraídos (100% sucesso)
🔧 Código limpo e otimizado
💾 Dados exportados em JSON
🚀 Pronto para implementar ponto 1.2

📝 RESUMO EXECUTIVO:
• Empresa: SCALA TRANSPORTE E ADMINISTRACAO LTDA.
• Funcionário: ANDRE LUIS DE MORAES DA ROSA
• Período: Maio/2025
• Status: Dados de cabeçalho extraídos com sucesso
